# History of tesselations 2 |  Manim Animation

This notebook builds the videos to create an animation-explainer of the history of tesselations.
It animates tiles or polygons to show how to build different tesselations.

This notebook animates the semi-regular polygons (as explained in ../tramo2/)

In [3]:
import manim as mn
from manim import *

config.media_width = "75%"
config.verbosity = "WARNING"

print(mn.__version__)

0.21.0


## Intro to semi-regular polygons

In [10]:
%%manim -qm CombinacionesPoligonos

class CombinacionesPoligonos(Scene):
    def construct(self):
        question = VGroup(
            Text(
                "¿Podemos cubrir el plano con combinaciones",
                font_size=32,
                color=WHITE,
            ),
            Text(
                "de polígonos regulares?",
                font_size=32,
                color=WHITE,
            ),
        ).arrange(DOWN, buff=0.2)
        if question.width > config.frame_width - 1:
            question.scale_to_fit_width(config.frame_width - 1)

        self.play(FadeIn(question), run_time=0.8)
        self.wait(1.5)

        subtitle = Text(
            "hay miles y miles de posibilidades!",
            font_size=36,
            color=WHITE,
        )
        if subtitle.width > config.frame_width - 1:
            subtitle.scale_to_fit_width(config.frame_width - 1)

        self.play(FadeOut(question), FadeIn(subtitle), run_time=1)
        self.wait(0.4)
        self.play(subtitle.animate.to_edge(UP, buff=0.45), run_time=0.6)

        colors = color_gradient(
            [
                "#FF8A80",
                "#FFD180",
                "#FFFF8D",
                "#B9F6CA",
                "#80D8FF",
                "#8C9EFF",
                "#EA80FC",
                "#FF80AB",
            ],
            24,
        )

        polys = VGroup()
        for n, color in zip(range(3, 27), colors):
            poly = RegularPolygon(
                n=n,
                radius=0.58,
                color=color,
                stroke_width=3,
                fill_color=color,
                fill_opacity=0.45,
            )
            if n == 4:
                poly.rotate(45 * DEGREES)
            polys.add(poly)

        polys.arrange_in_grid(rows=4, cols=6, buff=0.32)
        max_w = config.frame_width - 0.8
        max_h = subtitle.get_bottom()[1] + config.frame_height / 2 - 0.55
        if polys.width > max_w:
            polys.scale_to_fit_width(max_w)
        if polys.height > max_h:
            polys.scale_to_fit_height(max_h)
        polys.next_to(subtitle, DOWN, buff=0.4)

        polys.set_z_index(1)
        subtitle.set_z_index(2)

        self.play(
            LaggedStart(*[Create(poly) for poly in polys], lag_ratio=0.12),
            run_time=1.6,
        )
        self.wait(0.4)

        outgoing = []
        arrows_by_pair = {}
        for i, src in enumerate(polys):
            n_src = i + 3
            arrows_i = VGroup()
            start = src.get_center()
            for j in range(i + 1, len(polys)):
                n_dst = j + 3
                dst = polys[j]
                end_center = dst.get_center()
                vec = end_center - start
                nrm = vec / np.linalg.norm(vec)
                r_dst = np.linalg.norm(dst.get_vertices()[0] - end_center)
                end = end_center - nrm * (r_dst + 0.04)
                arrow = Arrow(
                    start,
                    end,
                    color=colors[i],
                    stroke_width=5,
                    stroke_opacity=0.9,
                    buff=0,
                    tip_length=0.18,
                    max_tip_length_to_length_ratio=0.2,
                    max_stroke_width_to_length_ratio=10,
                )
                arrow.set_z_index(0)
                arrows_i.add(arrow)
                arrows_by_pair[(n_src, n_dst)] = arrow
            if len(arrows_i) > 0:
                outgoing.append(arrows_i)

        self.play(
            LaggedStart(
                *[
                    LaggedStart(
                        *[Create(arrow) for arrow in group],
                        lag_ratio=0.06,
                    )
                    for group in outgoing
                ],
                lag_ratio=0.18,
            ),
            run_time=4,
        )
        self.wait(1.2)

        closing = Text(
            "pero solo 8 combinaciones no dejan huecos",
            font_size=36,
            color=WHITE,
        )
        if closing.width > config.frame_width - 1:
            closing.scale_to_fit_width(config.frame_width - 1)
        closing.to_edge(UP, buff=0.45)
        closing.set_z_index(2)

        large_polys = VGroup(
            *[poly for n, poly in zip(range(3, 27), polys) if n > 12]
        )
        large_arrows = VGroup(
            *[
                arrow
                for (n_src, n_dst), arrow in arrows_by_pair.items()
                if n_src > 12 or n_dst > 12
            ]
        )
        self.play(
            FadeOut(large_polys),
            FadeOut(large_arrows),
            FadeOut(subtitle),
            run_time=1.4,
        )
        self.wait(0.4)

        keep_links = {
            frozenset({3, 4}),   # triangle <-> square
            frozenset({4, 8}),   # octagon <-> square
            frozenset({3, 6}),   # triangle <-> hexagon
            frozenset({3, 12}),  # dodecagon <-> triangle
            frozenset({6, 12}),  # dodecagon <-> hexagon
            frozenset({4, 12}),  # dodecagon <-> square
            frozenset({4, 6}),   # hexagon <-> square
        }
        extra_arrows = VGroup(
            *[
                arrow
                for (n_src, n_dst), arrow in arrows_by_pair.items()
                if n_src <= 12
                and n_dst <= 12
                and frozenset({n_src, n_dst}) not in keep_links
            ]
        )
        unused_polys = VGroup(
            *[
                poly
                for n, poly in zip(range(3, 27), polys)
                if n in {5, 7, 9, 10, 11}
            ]
        )
        self.play(
            FadeOut(extra_arrows),
            FadeOut(unused_polys),
            FadeIn(closing),
            run_time=1.4,
        )
        self.wait(0.4)

        kept_polys = VGroup(
            *[
                poly
                for n, poly in zip(range(3, 27), polys)
                if n in {3, 4, 6, 8, 12}
            ]
        )
        see_which = Text("Veamos cuales son", font_size=32, color=WHITE)
        see_which.next_to(kept_polys, DOWN, buff=0.55)
        if see_which.width > config.frame_width - 1:
            see_which.scale_to_fit_width(config.frame_width - 1)
        self.play(FadeIn(see_which), run_time=0.8)
        self.wait(1.5)


Manim Community v0.21.0

## 2.1 Triángulos y Cuadrados

In [12]:
%%manim -qm TriangulosYCuadrados1

class TriangulosYCuadrados1(Scene):
    def construct(self):
        square_radius = 0.8
        triangle_radius = square_radius * np.sin(PI / 4) / np.sin(PI / 3)
        half_sq = square_radius * np.sin(PI / 4)
        dx_right = 4 * half_sq
        dx_up = half_sq
        dy_up = 2 * half_sq + 3 * triangle_radius * np.cos(PI / 3)
        start_x = config.frame_width / 2 + dx_right
        stroke_width = 4
        square_tilt = 45 * DEGREES
        triangle_tilt = 90 * DEGREES

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def move_centroid_to(mob, point):
            mob.shift(point - centroid(mob))

        def cell_origin(r, c):
            return RIGHT * (c * dx_right + r * dx_up) + UP * (r * dy_up)

        local = {
            "S0": ORIGIN,
            "S1": RIGHT * (2 * half_sq),
            "T1": UP * (triangle_radius * np.cos(PI / 3) + half_sq),
            "T2": RIGHT * half_sq
            + UP * (2 * triangle_radius * np.cos(PI / 3) + half_sq),
            "T3": RIGHT * (2 * half_sq)
            + UP * (triangle_radius * np.cos(PI / 3) + half_sq),
            "T4": RIGHT * (3 * half_sq)
            + UP * (2 * triangle_radius * np.cos(PI / 3) + half_sq),
        }
        pointing = {
            "T1": True,
            "T2": False,
            "T3": True,
            "T4": False,
        }

        def piece_center(r, c, name):
            return cell_origin(r, c) + local[name]

        def make_square():
            sq = RegularPolygon(
                n=4,
                radius=square_radius,
                color=WHITE,
                stroke_width=stroke_width,
            )
            sq.rotate(square_tilt, about_point=centroid(sq))
            return sq

        def make_triangle(pointing_up):
            tri = RegularPolygon(
                n=3,
                radius=triangle_radius,
                color=WHITE,
                stroke_width=stroke_width,
            )
            if not pointing_up:
                tri.rotate(PI, about_point=centroid(tri))
            return tri

        def make_piece(name):
            if name.startswith("S"):
                return make_square()
            return make_triangle(pointing[name])

        initial_specs = [
            (0, 0, "S0"),
            (0, 0, "S1"),
            (0, 0, "T1"),
            (0, 0, "T3"),
            (0, 0, "T2"),
            (0, 0, "T4"),
            (1, 0, "S0"),
            (1, 0, "S1"),
        ]
        initial_keys = set(initial_specs)
        initial_pos = [piece_center(r, c, name) for r, c, name in initial_specs]
        shift = -sum(initial_pos, ORIGIN) / len(initial_pos)
        initial = [
            (piece_center(r, c, name) + shift, name)
            for r, c, name in initial_specs
        ]

        label = Text("Triángulos y Cuadrados 1", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        for target, name in initial:
            mob = make_piece(name)
            tilt = square_tilt if name.startswith("S") else triangle_tilt
            mob.rotate(tilt, about_point=centroid(mob))

            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * square_radius)
            move_centroid_to(mob, start)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = centroid(mob).copy()

            def land(mob, alpha, _c0=c0, _target=target, _tilt=tilt):
                mob.restore()
                mob.shift(alpha * (_target - _c0))
                mob.rotate(
                    -_tilt * alpha,
                    about_point=centroid(mob),
                )

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        margin_x = config.frame_width / 2 + dx_right
        margin_y = config.frame_height / 2 + dy_up
        rest = []
        names = ["S0", "S1", "T1", "T2", "T3", "T4"]
        for r in range(-8, 10):
            for c in range(-10, 12):
                for name in names:
                    if (r, c, name) in initial_keys:
                        continue
                    pos = piece_center(r, c, name) + shift
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    mob = make_piece(name)
                    move_centroid_to(mob, pos)
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## 2.2 Triángulos y Cuadrados

In [13]:
%%manim -qm TriangulosYCuadrados2

class TriangulosYCuadrados2(Scene):
    def construct(self):
        size = 0.5
        v1 = size * np.array([0.7071067811865475, -2.6389584337646843, 0.0])
        v2 = size * np.array([2.6389584337646843, 0.7071067811865475, 0.0])
        rot_origin = size * np.array([-0.7071067811865476, -0.7071067811865475, 0.0])
        global_rot = 30 * DEGREES
        start_x = config.frame_width / 2 + 2.2
        stroke_width = 4
        square_tilt = 45 * DEGREES
        triangle_tilt = 90 * DEGREES

        unit_verts = {
            "S0": [
                (0.707107, -0.707107),
                (0.707107, 0.707107),
                (-0.707107, 0.707107),
                (-0.707107, -0.707107),
            ],
            "S1": [
                (1.931852, 1.414214),
                (1.224745, 2.638958),
                (0.000000, 1.931852),
                (0.707107, 0.707107),
            ],
            "T1": [
                (0.000000, 1.931852),
                (-0.707107, 0.707107),
                (0.707107, 0.707107),
            ],
            "T2": [
                (1.931852, 0.000000),
                (0.707107, 0.707107),
                (0.707107, -0.707107),
            ],
            "T3": [
                (1.931852, 1.414214),
                (0.707107, 0.707107),
                (1.931852, 0.000000),
            ],
            "T4": [
                (1.931852, 1.414214),
                (2.638958, 2.638958),
                (1.224745, 2.638958),
            ],
        }
        piece_names = ["S0", "T1", "T2", "S1", "T3", "T4"]

        def rotate_about(point, origin, angle):
            return origin + rotate_vector(point - origin, angle)

        def world_point(x, y, i, j):
            p = size * np.array([x, y, 0.0]) + i * v1 + j * v2
            return rotate_about(p, rot_origin, global_rot)

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def group_centroid(group):
            return sum((centroid(m) for m in group), ORIGIN) / len(group)

        def move_centroid_to(mob, point, center_fn=centroid):
            mob.shift(point - center_fn(mob))

        def make_piece(i, j, name):
            verts = [world_point(x, y, i, j) for x, y in unit_verts[name]]
            return Polygon(*verts, color=WHITE, stroke_width=stroke_width)

        def make_primitive(i, j):
            return VGroup(*[make_piece(i, j, name) for name in piece_names])

        def fly_in(mob, target, tilt, center_fn=centroid):
            mob.rotate(tilt, about_point=center_fn(mob))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * size)
            move_centroid_to(mob, start, center_fn)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = center_fn(mob).copy()

            def land(m, alpha, _c0=c0, _target=target, _tilt=tilt, _center_fn=center_fn):
                m.restore()
                m.shift(alpha * (_target - _c0))
                m.rotate(-_tilt * alpha, about_point=_center_fn(m))

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        initial_cells = [(0, 0), (1, 0), (2, 0), (0, 1), (1, 1), (2, 1)]
        initial_keys = set(initial_cells)

        sample = [centroid(make_piece(0, 0, name)) for name in piece_names]
        shift = -sum(sample, ORIGIN) / len(sample)

        label = Text("Triángulos y Cuadrados 2", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        # One primitive, tile by tile, then five more copies as units
        first = make_primitive(0, 0)
        first.shift(shift)
        for mob, name in zip(first, piece_names):
            tilt = square_tilt if name.startswith("S") else triangle_tilt
            fly_in(mob, centroid(mob), tilt)

        for i, j in initial_cells[1:]:
            group = make_primitive(i, j)
            group.shift(shift)
            fly_in(group, group_centroid(group), square_tilt, center_fn=group_centroid)

        margin_x = config.frame_width / 2 + 2.0
        margin_y = config.frame_height / 2 + 2.0
        rest = []
        for i in range(-12, 14):
            for j in range(-12, 14):
                if (i, j) in initial_keys:
                    continue
                for name in piece_names:
                    mob = make_piece(i, j, name)
                    mob.shift(shift)
                    pos = centroid(mob)
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## 2.3 Cuadrados y Octágonos

In [14]:
%%manim -qm CuadradosYOctagonos

class CuadradosYOctagonos(Scene):
    def construct(self):
        size = 0.42
        alpha = PI / 4
        square_radius = size
        octagon_radius = size * np.sin(alpha) / np.sin(alpha / 2)
        L = 2 * size * np.sin(alpha)
        dx = L * (1 + 1 / (2 * np.sin(alpha)))
        v1 = RIGHT * dx + UP * dx
        v2 = RIGHT * dx + DOWN * dx
        o_local = UP * (
            octagon_radius * np.cos(alpha / 2) + square_radius * np.cos(alpha)
        )
        start_x = config.frame_width / 2 + 2.4
        stroke_width = 4
        square_tilt = 45 * DEGREES
        octagon_tilt = 22.5 * DEGREES

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def group_centroid(group):
            return sum((centroid(m) for m in group), ORIGIN) / len(group)

        def move_centroid_to(mob, point, center_fn=centroid):
            mob.shift(point - center_fn(mob))

        def cell_origin(i, j):
            return i * v1 + j * v2

        def make_square():
            sq = RegularPolygon(
                n=4,
                radius=square_radius,
                color=WHITE,
                stroke_width=stroke_width,
            )
            sq.rotate(45 * DEGREES, about_point=centroid(sq))
            return sq

        def make_octagon():
            octagon = RegularPolygon(
                n=8,
                radius=octagon_radius,
                color=WHITE,
                stroke_width=stroke_width,
            )
            octagon.rotate(22.5 * DEGREES, about_point=centroid(octagon))
            return octagon

        def make_primitive(i, j):
            origin = cell_origin(i, j)
            sq = make_square()
            octagon = make_octagon()
            move_centroid_to(sq, origin)
            move_centroid_to(octagon, origin + o_local)
            return VGroup(sq, octagon)

        def fly_in(mob, target, tilt, center_fn=centroid):
            mob.rotate(tilt, about_point=center_fn(mob))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * octagon_radius)
            move_centroid_to(mob, start, center_fn)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = center_fn(mob).copy()

            def land(m, alpha_t, _c0=c0, _target=target, _tilt=tilt, _center_fn=center_fn):
                m.restore()
                m.shift(alpha_t * (_target - _c0))
                m.rotate(-_tilt * alpha_t, about_point=_center_fn(m))

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        initial_cells = [
            (0, 0),
            (1, 0),
            (0, 1),
            (-1, 0),
            (0, -1),
            (1, 1),
            (-1, 1),
            (1, -1),
        ]
        initial_keys = set(initial_cells)

        first = make_primitive(0, 0)
        shift = -group_centroid(first)
        first.shift(shift)

        label = Text("Cuadrados y Octágonos", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        fly_in(first[0], centroid(first[0]), square_tilt)
        fly_in(first[1], centroid(first[1]), octagon_tilt)

        for i, j in initial_cells[1:]:
            group = make_primitive(i, j)
            group.shift(shift)
            fly_in(group, group_centroid(group), square_tilt, center_fn=group_centroid)

        margin_x = config.frame_width / 2 + 2.2
        margin_y = config.frame_height / 2 + 2.2
        rest = []
        for i in range(-10, 12):
            for j in range(-10, 12):
                if (i, j) in initial_keys:
                    continue
                group = make_primitive(i, j)
                group.shift(shift)
                for mob in group:
                    pos = centroid(mob)
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## 2.4 Hexágonos y Triángulos

In [16]:
%%manim -qm HexagonosYTriangulos

class HexagonosYTriangulos(Scene):
    def construct(self):
        size = 0.5
        tri_r = size * np.sin(PI / 6) / np.sin(PI / 3)
        s60 = np.sin(PI / 3)
        c30 = np.cos(PI / 6)
        v1 = RIGHT * (4 * tri_r * s60) + UP * (2 * size * np.sin(4 * PI / 6))
        v2 = RIGHT * (5 * tri_r * s60) + DOWN * (size * np.sin(4 * PI / 6))
        start_x = config.frame_width / 2 + 2.4
        stroke_width = 4
        hex_tilt = 30 * DEGREES
        triangle_tilt = 90 * DEGREES

        local = {
            "H0": ORIGIN,
            "T1": LEFT * (2 * tri_r * s60) + DOWN * tri_r,
            "T2": LEFT * (2 * tri_r * s60) + DOWN * (2 * tri_r),
            "T3": LEFT * (tri_r * s60) + DOWN * (tri_r + size * c30),
            "T4": DOWN * (tri_r / 2 + size * c30),
            "T5": RIGHT * (tri_r * s60) + DOWN * (tri_r + size * c30),
            "T6": RIGHT * (2 * tri_r * s60) + DOWN * (2 * tri_r),
            "T7": RIGHT * (2 * tri_r * s60) + DOWN * tri_r,
            "T8": RIGHT * (3 * tri_r * s60) + DOWN * (tri_r / 2),
        }
        pointing_up = {
            "T1": True,
            "T2": False,
            "T3": True,
            "T4": False,
            "T5": True,
            "T6": False,
            "T7": True,
            "T8": False,
        }
        piece_names = ["H0", "T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8"]

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def group_centroid(group):
            return sum((centroid(m) for m in group), ORIGIN) / len(group)

        def move_centroid_to(mob, point, center_fn=centroid):
            mob.shift(point - center_fn(mob))

        def cell_origin(i, j):
            return i * v1 + j * v2

        def make_hexagon():
            return RegularPolygon(
                n=6,
                radius=size,
                color=WHITE,
                stroke_width=stroke_width,
            )

        def make_triangle(up):
            tri = RegularPolygon(
                n=3,
                radius=tri_r,
                color=WHITE,
                stroke_width=stroke_width,
            )
            if not up:
                tri.rotate(PI, about_point=centroid(tri))
            return tri

        def make_piece(name):
            if name == "H0":
                return make_hexagon()
            return make_triangle(pointing_up[name])

        def make_primitive(i, j):
            origin = cell_origin(i, j)
            group = VGroup()
            for name in piece_names:
                mob = make_piece(name)
                move_centroid_to(mob, origin + local[name])
                group.add(mob)
            return group

        def fly_in(mob, target, tilt, center_fn=centroid):
            mob.rotate(tilt, about_point=center_fn(mob))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * size)
            move_centroid_to(mob, start, center_fn)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = center_fn(mob).copy()

            def land(m, alpha, _c0=c0, _target=target, _tilt=tilt, _center_fn=center_fn):
                m.restore()
                m.shift(alpha * (_target - _c0))
                m.rotate(-_tilt * alpha, about_point=_center_fn(m))

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        initial_cells = [
            (0, 0),
            (1, 0),
            (0, 1),
            (-1, 1),
            (-1, 0),
            (0, -1),
        ]
        initial_keys = set(initial_cells)

        first = make_primitive(0, 0)
        shift = -group_centroid(first)
        first.shift(shift)

        label = Text("Hexágonos y Triángulos", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        fly_in(first[0], centroid(first[0]), hex_tilt)
        for mob, name in zip(first[1:], piece_names[1:]):
            fly_in(mob, centroid(mob), triangle_tilt)

        for i, j in initial_cells[1:]:
            group = make_primitive(i, j)
            group.shift(shift)
            fly_in(group, group_centroid(group), hex_tilt, center_fn=group_centroid)

        margin_x = config.frame_width / 2 + 2.2
        margin_y = config.frame_height / 2 + 2.2
        rest = []
        for i in range(-10, 12):
            for j in range(-10, 12):
                if (i, j) in initial_keys:
                    continue
                group = make_primitive(i, j)
                group.shift(shift)
                for mob in group:
                    pos = centroid(mob)
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## 2.5 Dodecágonos y Triángulos


In [20]:
%%manim -qm DodecagonosYTriangulos

class DodecagonosYTriangulos(Scene):
    def construct(self):
        size = 0.8
        tri_r = size * np.sin(PI / 12) / np.sin(PI / 3)
        d = size * np.cos(PI / 12) + tri_r * np.cos(PI / 3)
        v1 = UP * (2 * size * np.cos(PI / 12))
        v2 = (
            RIGHT * (2 * size * np.cos(PI / 12) * np.cos(PI / 6))
            + UP * (size * np.cos(PI / 12))
        )
        start_x = config.frame_width / 2 + 2.4
        stroke_width = 6
        dodeca_tilt = 15 * DEGREES
        triangle_tilt = 90 * DEGREES

        local = {
            "D0": ORIGIN,
            "T1": RIGHT * (d * np.cos(PI / 3)) + UP * (d * np.sin(PI / 3)),
            "T2": LEFT * (d * np.cos(PI / 3)) + UP * (d * np.sin(PI / 3)),
        }
        tri_rot = {
            "T1": -60 * DEGREES,
            "T2": 0,
        }
        piece_names = ["D0", "T1", "T2"]

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def group_centroid(group):
            return sum((centroid(m) for m in group), ORIGIN) / len(group)

        def move_centroid_to(mob, point, center_fn=centroid):
            mob.shift(point - center_fn(mob))

        def cell_origin(i, j):
            return i * v1 + j * v2

        def make_dodecagon():
            dodeca = RegularPolygon(
                n=12,
                radius=size,
                color=WHITE,
                stroke_width=stroke_width,
            )
            dodeca.rotate(45 * DEGREES, about_point=centroid(dodeca))
            return dodeca

        def make_triangle(rot):
            tri = RegularPolygon(
                n=3,
                radius=tri_r,
                start_angle=0,
                color=WHITE,
                stroke_width=stroke_width,
            )
            if rot:
                tri.rotate(rot, about_point=centroid(tri))
            return tri

        def make_piece(name):
            if name == "D0":
                return make_dodecagon()
            return make_triangle(tri_rot[name])

        def make_primitive(i, j):
            origin = cell_origin(i, j)
            group = VGroup()
            for name in piece_names:
                mob = make_piece(name)
                move_centroid_to(mob, origin + local[name])
                group.add(mob)
            return group

        def fly_in(mob, target, tilt, center_fn=centroid):
            mob.rotate(tilt, about_point=center_fn(mob))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * size)
            move_centroid_to(mob, start, center_fn)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = center_fn(mob).copy()

            def land(m, alpha, _c0=c0, _target=target, _tilt=tilt, _center_fn=center_fn):
                m.restore()
                m.shift(alpha * (_target - _c0))
                m.rotate(-_tilt * alpha, about_point=_center_fn(m))

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        initial_cells = [
            (0, 0),
            (1, 0),
            (0, 1),
            (-1, 1),
            (-1, 0),
            (0, -1),
        ]
        initial_keys = set(initial_cells)

        first = make_primitive(0, 0)
        shift = -group_centroid(first)
        first.shift(shift)

        label = Text("Dodecágonos y Triángulos", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        fly_in(first[0], centroid(first[0]), dodeca_tilt)
        for mob, name in zip(first[1:], piece_names[1:]):
            fly_in(mob, centroid(mob), triangle_tilt)

        for i, j in initial_cells[1:]:
            group = make_primitive(i, j)
            group.shift(shift)
            fly_in(group, group_centroid(group), dodeca_tilt, center_fn=group_centroid)

        margin_x = config.frame_width / 2 + 2.2
        margin_y = config.frame_height / 2 + 2.2
        rest = []
        for i in range(-10, 12):
            for j in range(-10, 12):
                if (i, j) in initial_keys:
                    continue
                group = make_primitive(i, j)
                group.shift(shift)
                for mob in group:
                    pos = centroid(mob)
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## 2.6 Hexágonos y Triángulos 2

In [22]:
%%manim -qm HexagonosYTriangulos2

class HexagonosYTriangulos2(Scene):
    def construct(self):
        size = 0.5
        tri_r = size * np.sin(PI / 6) / np.sin(PI / 3)
        d = size * np.cos(PI / 6) + tri_r * np.cos(PI / 3)
        v1 = UP * (2 * size)
        v2 = (
            RIGHT * (2 * size * np.cos(PI / 6))
            + UP * (2 * size * np.sin(PI / 6))
        )
        start_x = config.frame_width / 2 + 2.4
        stroke_width = 4
        hex_tilt = 30 * DEGREES
        triangle_tilt = 90 * DEGREES

        local = {
            "H0": ORIGIN,
            "T1": RIGHT * (d * np.cos(PI / 3)) + UP * (d * np.sin(PI / 3)),
            "T2": LEFT * (d * np.cos(PI / 3)) + UP * (d * np.sin(PI / 3)),
        }
        tri_rot = {
            "T1": -60 * DEGREES,
            "T2": 0,
        }
        piece_names = ["H0", "T1", "T2"]

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def group_centroid(group):
            return sum((centroid(m) for m in group), ORIGIN) / len(group)

        def move_centroid_to(mob, point, center_fn=centroid):
            mob.shift(point - center_fn(mob))

        def cell_origin(i, j):
            return i * v1 + j * v2

        def make_hexagon():
            hexagon = RegularPolygon(
                n=6,
                radius=size,
                color=WHITE,
                stroke_width=stroke_width,
            )
            hexagon.rotate(30 * DEGREES, about_point=centroid(hexagon))
            return hexagon

        def make_triangle(rot):
            tri = RegularPolygon(
                n=3,
                radius=tri_r,
                start_angle=0,
                color=WHITE,
                stroke_width=stroke_width,
            )
            if rot:
                tri.rotate(rot, about_point=centroid(tri))
            return tri

        def make_piece(name):
            if name == "H0":
                return make_hexagon()
            return make_triangle(tri_rot[name])

        def make_primitive(i, j):
            origin = cell_origin(i, j)
            group = VGroup()
            for name in piece_names:
                mob = make_piece(name)
                move_centroid_to(mob, origin + local[name])
                group.add(mob)
            return group

        def fly_in(mob, target, tilt, center_fn=centroid):
            mob.rotate(tilt, about_point=center_fn(mob))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * size)
            move_centroid_to(mob, start, center_fn)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = center_fn(mob).copy()

            def land(m, alpha, _c0=c0, _target=target, _tilt=tilt, _center_fn=center_fn):
                m.restore()
                m.shift(alpha * (_target - _c0))
                m.rotate(-_tilt * alpha, about_point=_center_fn(m))

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        initial_cells = [
            (0, 0),
            (1, 0),
            (0, 1),
            (-1, 1),
            (-1, 0),
            (0, -1),
            (1, -1),
        ]
        initial_keys = set(initial_cells)

        first = make_primitive(0, 0)
        shift = -group_centroid(first)
        first.shift(shift)

        label = Text("Hexágonos y Triángulos 2", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        fly_in(first[0], centroid(first[0]), hex_tilt)
        for mob, name in zip(first[1:], piece_names[1:]):
            fly_in(mob, centroid(mob), triangle_tilt)

        for i, j in initial_cells[1:]:
            group = make_primitive(i, j)
            group.shift(shift)
            fly_in(group, group_centroid(group), hex_tilt, center_fn=group_centroid)

        margin_x = config.frame_width / 2 + 2.2
        margin_y = config.frame_height / 2 + 2.2
        rest = []
        for i in range(-10, 12):
            for j in range(-10, 12):
                if (i, j) in initial_keys:
                    continue
                group = make_primitive(i, j)
                group.shift(shift)
                for mob in group:
                    pos = centroid(mob)
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## 2.7 Dodecágonos, Cuadrados y Hexágonos

In [24]:
%%manim -qm DodecagonosCuadradosYHexagonos

class DodecagonosCuadradosYHexagonos(Scene):
    def construct(self):
        size = 0.8
        hex_r = size * np.sin(PI / 12) / np.sin(PI / 6)
        sq_r = size * np.sin(PI / 12) / np.sin(PI / 4)
        d_sq = size * np.cos(PI / 12) + sq_r * np.cos(PI / 4)
        d_hex = size * np.cos(PI / 12) + hex_r * np.cos(PI / 6)
        v1 = RIGHT * (2 * d_sq)
        v2 = (
            RIGHT * d_sq
            + UP * (
                sq_r * np.cos(PI / 4)
                + 2 * hex_r * np.cos(PI / 6)
                + size * np.cos(PI / 12)
            )
        )
        start_x = config.frame_width / 2 + 2.4
        stroke_width = 4
        dodeca_tilt = 15 * DEGREES
        square_tilt = 45 * DEGREES
        hex_tilt = 30 * DEGREES

        local = {
            "D0": ORIGIN,
            "S1": RIGHT * (d_sq * np.sin(PI / 6)) + DOWN * (d_sq * np.cos(PI / 6)),
            "S2": LEFT * (d_sq * np.sin(PI / 6)) + DOWN * (d_sq * np.cos(PI / 6)),
            "S3": RIGHT * d_sq,
            "H0": DOWN * d_hex,
            "H1": RIGHT * (d_hex * np.sin(PI / 3)) + DOWN * (d_hex * np.cos(PI / 3)),
        }
        rest_rot = {
            "D0": 45 * DEGREES,
            "S1": -15 * DEGREES,
            "S2": -75 * DEGREES,
            "S3": 45 * DEGREES,
            "H0": 0,
            "H1": 0,
        }
        fly_tilt = {
            "D0": dodeca_tilt,
            "S1": square_tilt,
            "S2": square_tilt,
            "S3": square_tilt,
            "H0": hex_tilt,
            "H1": hex_tilt,
        }
        sides = {"D0": 12, "S1": 4, "S2": 4, "S3": 4, "H0": 6, "H1": 6}
        radii = {
            "D0": size,
            "S1": sq_r,
            "S2": sq_r,
            "S3": sq_r,
            "H0": hex_r,
            "H1": hex_r,
        }
        piece_names = ["D0", "S1", "S2", "S3", "H0", "H1"]

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def group_centroid(group):
            return sum((centroid(m) for m in group), ORIGIN) / len(group)

        def move_centroid_to(mob, point, center_fn=centroid):
            mob.shift(point - center_fn(mob))

        def cell_origin(i, j):
            return i * v1 + j * v2

        def make_piece(name):
            mob = RegularPolygon(
                n=sides[name],
                radius=radii[name],
                start_angle=0,
                color=WHITE,
                stroke_width=stroke_width,
            )
            rot = rest_rot[name]
            if rot:
                mob.rotate(rot, about_point=centroid(mob))
            return mob

        def make_primitive(i, j):
            origin = cell_origin(i, j)
            group = VGroup()
            for name in piece_names:
                mob = make_piece(name)
                move_centroid_to(mob, origin + local[name])
                group.add(mob)
            return group

        def fly_in(mob, target, tilt, center_fn=centroid):
            mob.rotate(tilt, about_point=center_fn(mob))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * size)
            move_centroid_to(mob, start, center_fn)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = center_fn(mob).copy()

            def land(m, alpha, _c0=c0, _target=target, _tilt=tilt, _center_fn=center_fn):
                m.restore()
                m.shift(alpha * (_target - _c0))
                m.rotate(-_tilt * alpha, about_point=_center_fn(m))

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        initial_cells = [
            (0, 0),
            (1, 0),
            (0, 1),
            (-1, 1),
            (-1, 0),
            (0, -1),
        ]
        initial_keys = set(initial_cells)

        first = make_primitive(0, 0)
        shift = -group_centroid(first)
        first.shift(shift)

        label = Text("Dodecágonos, Cuadrados y Hexágonos", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        for mob, name in zip(first, piece_names):
            fly_in(mob, centroid(mob), fly_tilt[name])

        for i, j in initial_cells[1:]:
            group = make_primitive(i, j)
            group.shift(shift)
            fly_in(group, group_centroid(group), dodeca_tilt, center_fn=group_centroid)

        margin_x = config.frame_width / 2 + 2.2
        margin_y = config.frame_height / 2 + 2.2
        rest = []
        for i in range(-10, 12):
            for j in range(-10, 12):
                if (i, j) in initial_keys:
                    continue
                group = make_primitive(i, j)
                group.shift(shift)
                for mob in group:
                    pos = centroid(mob)
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## 2.8 Triángulos, Cuadrados y Hexágonos

In [ ]:
%%manim -qm TriangulosCuadradosYHexagonos

class TriangulosCuadradosYHexagonos(Scene):
    def construct(self):
        size = 0.5
        sq_r = size * np.sin(PI / 6) / np.sin(PI / 4)
        tri_r = sq_r * np.sin(PI / 4) / np.sin(PI / 3)
        hx = size * np.cos(PI / 6)
        hs = sq_r * np.cos(PI / 4)
        ht = tri_r * (1 + np.sin(PI / 6))
        d_sq = hx + hs
        d_tri = size + tri_r
        v1 = UP * (2 * hx + 2 * hs)
        v2 = RIGHT * (size + ht + hs) + UP * (hx + hs)
        start_x = config.frame_width / 2 + 2.4
        stroke_width = 4
        hex_tilt = 30 * DEGREES
        square_tilt = 45 * DEGREES
        triangle_tilt = 90 * DEGREES

        local = {
            "H1": ORIGIN,
            "S1": DOWN * d_sq,
            "S2": LEFT * (d_sq * np.sin(PI / 3)) + DOWN * (d_sq * np.cos(PI / 3)),
            "S3": RIGHT * (d_sq * np.sin(PI / 3)) + DOWN * (d_sq * np.cos(PI / 3)),
            "T1": LEFT * (d_tri * np.cos(PI / 3)) + DOWN * (d_tri * np.sin(PI / 3)),
            "T2": RIGHT * (d_tri * np.cos(PI / 3)) + DOWN * (d_tri * np.sin(PI / 3)),
        }
        rest_rot = {
            "H1": 0,
            "S1": 45 * DEGREES,
            "S2": 165 * DEGREES,
            "S3": -75 * DEGREES,
            "T1": 60 * DEGREES,
            "T2": 0,
        }
        fly_tilt = {
            "H1": hex_tilt,
            "S1": square_tilt,
            "S2": square_tilt,
            "S3": square_tilt,
            "T1": triangle_tilt,
            "T2": triangle_tilt,
        }
        sides = {"H1": 6, "S1": 4, "S2": 4, "S3": 4, "T1": 3, "T2": 3}
        radii = {
            "H1": size,
            "S1": sq_r,
            "S2": sq_r,
            "S3": sq_r,
            "T1": tri_r,
            "T2": tri_r,
        }
        piece_names = ["H1", "S1", "S2", "S3", "T1", "T2"]

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def group_centroid(group):
            return sum((centroid(m) for m in group), ORIGIN) / len(group)

        def move_centroid_to(mob, point, center_fn=centroid):
            mob.shift(point - center_fn(mob))

        def cell_origin(i, j):
            return i * v1 + j * v2

        def make_piece(name):
            mob = RegularPolygon(
                n=sides[name],
                radius=radii[name],
                start_angle=0,
                color=WHITE,
                stroke_width=stroke_width,
            )
            rot = rest_rot[name]
            if rot:
                mob.rotate(rot, about_point=centroid(mob))
            return mob

        def make_primitive(i, j):
            origin = cell_origin(i, j)
            group = VGroup()
            for name in piece_names:
                mob = make_piece(name)
                move_centroid_to(mob, origin + local[name])
                group.add(mob)
            return group

        def fly_in(mob, target, tilt, center_fn=centroid):
            mob.rotate(tilt, about_point=center_fn(mob))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * size)
            move_centroid_to(mob, start, center_fn)
            self.add(mob)

            self.play(
                mob.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            mob.save_state()
            c0 = center_fn(mob).copy()

            def land(m, alpha, _c0=c0, _target=target, _tilt=tilt, _center_fn=center_fn):
                m.restore()
                m.shift(alpha * (_target - _c0))
                m.rotate(-_tilt * alpha, about_point=_center_fn(m))

            self.play(
                UpdateFromAlphaFunc(mob, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        initial_cells = [
            (0, 0),
            (1, 0),
            (0, 1),
            (-1, 1),
            (-1, 0),
            (0, -1),
        ]
        initial_keys = set(initial_cells)

        first = make_primitive(0, 0)
        shift = -group_centroid(first)
        first.shift(shift)

        label = Text("Triángulos, Cuadrados y Hexágonos", font_size=40, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        for mob, name in zip(first, piece_names):
            fly_in(mob, centroid(mob), fly_tilt[name])

        for i, j in initial_cells[1:]:
            group = make_primitive(i, j)
            group.shift(shift)
            fly_in(group, group_centroid(group), hex_tilt, center_fn=group_centroid)

        margin_x = config.frame_width / 2 + 2.2
        margin_y = config.frame_height / 2 + 2.2
        rest = []
        for i in range(-10, 12):
            for j in range(-10, 12):
                if (i, j) in initial_keys:
                    continue
                group = make_primitive(i, j)
                group.shift(shift)
                for mob in group:
                    pos = centroid(mob)
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    rest.append(mob)

        rest.sort(key=lambda m: np.linalg.norm(centroid(m)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(m) for m in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()
